In [6]:
import os
os.chdir("/Users/debapriyaroy/Documents/GitHub/Manufacturing-Performance-Analytics")

In [7]:
import pandas as pd

prod = pd.read_csv("data/raw/mes_production_hourly.csv")
prod["Line"].value_counts()

Line
L4         1992
L1         1987
L2         1919
L3         1912
Line 2      924
Line 3      906
Line 1      878
Line 4      865
LINE-03     567
LINE-04     547
LINE-02     544
LINE-01     520
line2       194
line3       187
line1       186
line4       172
Name: count, dtype: int64

## Data issues log

**Line names inconsistent** (mes_production_hourly, `Line` column): the 4 lines are written 4 ways each (`L1`, `Line 1`, `LINE-01`, `line1`), giving 16 values instead of 4.

In [8]:
prod["Shift_Code"].value_counts()


Shift_Code
A    4640
B    4632
C    3851
b     414
a     410
c     353
Name: count, dtype: int64

2. **Shift codes in mixed case** (mes_production_hourly, `Shift_Code` column): `a`, `b`, `c` appear next to `A`, `B`, `C`, giving 6 values instead of 3.

In [9]:
int(prod.duplicated().sum())

60

3. **Duplicate rows** (mes_production_hourly): 60 rows are exact copies of another row.

In [10]:
print(prod.dtypes)
prod["Hour_Start"].head()

Line                  str
Hour_Start            str
Shift_Code            str
Material_No           str
Sched_Min           int64
Planned_Stop_Min    int64
Downtime_Min        int64
Run_Min             int64
Total_Count         int64
dtype: object


0    28/05/2026 12:00
1    25/08/2026 07:00
2    15/04/2026 00:00
3    29/07/2026 15:00
4    18/05/2026 13:00
Name: Hour_Start, dtype: str

4. **Dates stored as text** (mes_production_hourly, `Hour_Start`): type is `str`, format is day-first `dd/mm/yyyy HH:MM`.
5. **Night shift crosses midnight** (mes_production_hourly): hours 00:00–05:59 belong to the previous production day (day runs 06:00–06:00), but there is no production-date column.

In [11]:
hour = pd.to_datetime(prod["Hour_Start"], format="%d/%m/%Y %H:%M").dt.hour
pd.crosstab(hour, prod["Shift_Code"].str.upper())

Shift_Code,A,B,C
Hour_Start,,,
0,0,0,524
1,0,0,526
2,0,0,526
3,0,0,524
4,0,0,525
5,0,0,527
6,631,0,0
7,632,0,0
8,636,0,0


In [12]:

stops = pd.read_csv("data/raw/mes_downtime_events.csv")
stops.head(10)

,Event_ID,Line,Machine_ID,Start_Time,End_Time,Duration_Min_Reported,Reason_Code,Operator_Comment
0,DT100000,L3,L3-TS,2026-04-01 06:00:00,2026-04-01 06:38:00,NaN,CO-01,Changeover to next SKU
1,DT100002,L2,L2-VFFS,2026-04-01 06:00:00,2026-04-01 07:17:00,77.0,CO-01,"SKU change, parts swap"
2,DT950021,L3,L3-TS,2026-04-01 06:00:00,2026-04-01 05:55:00,NaN,MS-01,clock sync error
3,DT100001,L4,L4-FL,2026-04-01 06:00:00,2026-04-01 06:24:00,24.0,CO-01,Format change + line clearance
4,DT100003,L3,L3-CW,2026-04-01 07:08:00,2026-04-01 07:15:00,7.0,MS-01,Packs bunching on conveyor
5,DT100004,L3,L3-TS,2026-04-01 07:17:00,2026-04-01 07:24:00,NaN,CO-02,New film reel loaded
6,DT100005,Line 1,L1-CP,2026-04-01 07:21:00,2026-04-01 07:28:00,7.0,MS-01,Product jam
7,DT100006,LINE-04,L4-FL,2026-04-01 07:27:00,2026-04-01 07:43:00,16.0,MT-01,Waiting for caps/jars
8,DT100007,L2,L2-VFFS,2026-04-01 07:29:00,2026-04-01 07:36:00,7.0,MS-01,NaN
9,DT100008,L2,L2-VFFS,2026-04-01 08:14:00,2026-04-01 08:19:00,NaN,CO-02,Film reel splice


In [13]:
start = pd.to_datetime(stops["Start_Time"])
end = pd.to_datetime(stops["End_Time"])
(end < start).sum()

np.int64(25)

6. **Stops ending before they start** (mes_downtime_events): 25 rows where End_Time is earlier than Start_Time (comment: "clock sync error").
   

In [14]:
stops.isna().sum()

Event_ID                    0
Line                        0
Machine_ID                  0
Start_Time                  0
End_Time                   82
Duration_Min_Reported    3418
Reason_Code                 0
Operator_Comment         5019
dtype: int64

7. **Stops with no end time** (mes_downtime_events, `End_Time`): 82 empty. Need to rebuild end time from reported duration.
8. **Reported duration often empty** (mes_downtime_events, `Duration_Min_Reported`): 3,418 empty (~25%). Need to use
end minus start instead.


In [15]:
stops.duplicated().sum()

np.int64(0)

In [16]:
stops.duplicated(subset=["Line", "Machine_ID", "Start_Time", "Reason_Code"]).sum()

np.int64(101)

9. **Same stop logged twice under a new ID** (mes_downtime_events): duplicated() finds 0, but matching on Line, Machine_ID, Start_Time and Reason_Code finds 101.

In [17]:
rejects = pd.read_csv("data/raw/qms_rejects_by_shift.csv")
rejects.head(10)

,Prod_Date,Shift,Line,Material_No,Seal_Integrity,Weight_Out_Of_Spec,Label_Code_Error,Metal_Detect,Damaged_Pack
0,2026-04-01,A,LINE-01,SB-101,125,459,65,14,81
1,2026-04-01,B,L1,SB-104,123,443,68,8,106
2,2026-04-01,C,Line 1,SB-104,91,395,60,11,115
3,2026-04-01,A,L2,PC-204,75,115,31,9,29
4,2026-04-01,B,line2,PC-201,95,177,41,17,34
5,2026-04-01,C,Line 2,PC-204,93,165,37,19,46
6,2026-04-01,A,line3,RM-302,82,50,37,1,34
7,2026-04-01,B,LINE-03,RM-303,103,69,57,2,39
8,2026-04-01,C,Line 3,RM-303,103,73,69,1,59
9,2026-04-01,A,LINE-04,SJ-402,47,120,74,6,61


10. **Line names written in different ways** (qms_rejects_by_shift): e.g. LINE-01, L1 and Line 1 are the same line. Same problem as issue 1.

11. **One column per reject reason** (qms_rejects_by_shift): reasons are spread across 5 columns (wide format). Needs reshaping into Reason and Packs columns (long format) before analysis.

In [18]:
products = pd.read_excel("data/raw/erp_material_master.xlsx")
products.head()

,Harvest Lane Foods - SAP material master extract,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Plant 2100 Riverside | extracted 2026-10-01,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Material_No,Material_Description,Pack_Format,Prod_Line,Rated_Speed_PPM,Std_Cost_USD,Net_Price_USD
3,SB-101,Oat & Honey Bar 35g,Flow-wrap single,L1,160,0.21,0.58
4,SB-102,Dark Choc Protein Bar 45g,Flow-wrap single,L1,150,0.33,0.79


In [19]:
products = pd.read_excel("data/raw/erp_material_master.xlsx", skiprows=3)
products.head()

,Material_No,Material_Description,Pack_Format,Prod_Line,Rated_Speed_PPM,Std_Cost_USD,Net_Price_USD
0,SB-101,Oat & Honey Bar 35g,Flow-wrap single,L1,160,0.21,0.58
1,SB-102,Dark Choc Protein Bar 45g,Flow-wrap single,L1,150,0.33,0.79
2,SB-103,Nut Butter Bar 40g,Flow-wrap single,L1,150,0.29,0.71
3,SB-104,Kids Fruit Bar 25g,Flow-wrap single,L1,170,0.14,0.45
4,PC-201,Trail Mix Pouch 200g,Pillow pouch,L2,80,1.05,2.05


12. **Title rows above the table** (erp_material_master.xlsx): the first 3 lines are a title, a note and a blank row, so pandas picked the wrong column names. Fixed by reading with skiprows=3.

In [20]:
products["Net_Price_USD"] - products["Std_Cost_USD"]

0     0.37
1     0.46
2     0.42
3     0.31
4     1.00
5     1.15
6     1.15
7     0.90
8     1.35
9     1.65
10    1.55
11    1.40
12    1.65
13    0.83
14    0.84
15    1.10
16    0.90
dtype: float64

13. **No profit column** (erp_material_master.xlsx): the file has cost and price but no profit per pack. Calculated it as Net_Price_USD − Std_Cost_USD.

In [21]:
pd.read_csv("data/raw/mes_downtime_events.csv")["Line"].value_counts()

Line
L1         2555
L3         2165
L2         1801
Line 1     1160
L4         1090
Line 3      913
Line 2      838
LINE-01     718
LINE-03     529
LINE-02     513
Line 4      512
LINE-04     326
line1       195
line3       170
line2       160
line4        96
Name: count, dtype: int64

14. **Line names messy in downtime file too:**  16 spellings (L1 / Line 1 / LINE-01 / line1 for each of the 4 lines). Same problem in all 3 files, so one mapping fixes all of them.